# Combined Muszyna genealogy: analytical notebook

## tl;dr

- The registry currently contains **164 people**, **63 family records / 60 recorded two-partner unions**, and **9 generation levels** on the longest documented parent–child path.
- Date coverage is the main analytical constraint: **77 people (47.0%)** have a birth year, **42 (25.6%)** have a death year, and only **42 (25.6%)** have both. Every time- or age-based view below reports its analyzed count and coverage.
- The parent–child graph has **26 components** because **25 people are isolated from recorded parent–child links**; the largest connected component contains **139 people**. Spouse links are deliberately excluded from that component definition.
- The notebook describes the **recorded genealogy**, not a representative population. “No recorded child,” “terminal line,” and sparse spouse ancestry are research states—not proof of childlessness, extinction, or shallow ancestry.
- Birthplace, residence, sex, explicit marriage dates, privacy status, and explicit unknown/not-applicable/inferred status fields are **not represented in the current analytical schema**. Those requested analyses are reported as unavailable or as clearly labeled proxies rather than inferred.

## Context & Methods

The editable CSV files in `data/registries/` are authoritative. This notebook reads the normalized, reproducible DuckDB artifact at `artifacts/analytics/genealogy_analytics.duckdb`, which is generated from those registries. It does not alter either source.

### Key assumptions and bias controls

| Situation | Treatment in this notebook |
|---|---|
| Unknown value | Kept missing and excluded only from calculations that require it; never converted to zero. |
| Not applicable | Not encoded explicitly in the current schema; no claim is made when eligibility cannot be established. |
| Inferred value | No new facts are inferred. Approximate years already recorded in the registry remain included and are noted as recorded approximations. |
| Recorded value | Used as documented in the registry/database. |
| Living/private | Not encoded explicitly; a missing death year is never treated as living. |
| Minimal spouse branch | `spouse_ancestry` is shown separately at the family-record level; sparse spouse ancestry is treated as project design, not inferior historical documentation. |

Definitions used below:

- **Generation** is the longest documented acyclic parent–child path assigned by the project builder; roots are generation 1.
- **Branch** means normalized `family_name_group` for people, or `branch_role` for family records. These are analytical groupings, not claims about biological lineage.
- **Founder** means a person with no recorded parent link. Descendant counts use documented directed edges only.
- **Terminal in the record** means no documented child link. It does not mean a biological line became extinct.
- **Recorded children** is used throughout because incomplete research can resemble low fertility.
- A family’s **child-based decade proxy** is the first recorded child’s birth decade. It is not a marriage date.
- A **reasonably complete child-date family** is a family with at least two recorded children and a known birth year for every recorded child. This screens for analyzable spacing, not true historical completeness.
- Longevity cohort comparisons use births through **1925** (at least 100 years before 2026) to reduce right-censoring. Missing death years remain missing.

## Data

### 1. Setup and source checks

Run this notebook from the repository root or from `analytics/`. A compatible ad-hoc environment can be started with:

```bash
uv run --with jupyter --with plotly jupyter lab analytics/genealogy_analytics.ipynb
```

In [1]:
from pathlib import Path
from collections import defaultdict, deque
from html import escape
import math

import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import HTML, display

pio.renderers.default = "notebook_connected"

def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "registries" / "people.csv").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate data/registries/people.csv from the current directory")

ROOT = find_repo_root()
DB_PATH = ROOT / "artifacts" / "analytics" / "genealogy_analytics.duckdb"
assert DB_PATH.is_file(), f"Missing {DB_PATH}; run uv run python -m scripts.build_genealogy_analytics"
con = duckdb.connect(str(DB_PATH), read_only=True)

PALETTE = {
    "blue": "#315C8C", "gold": "#C79A36", "orange": "#C96B3B",
    "olive": "#718355", "pink": "#B56576", "charcoal": "#27313A",
    "light": "#E9EDF2", "grid": "#D8DEE6", "muted": "#6B7280"
}

def query(sql, params=None):
    result = con.execute(sql, params or [])
    names = [d[0] for d in result.description]
    return [dict(zip(names, row)) for row in result.fetchall()]

def table(rows, columns=None, max_rows=25):
    if not rows:
        display(HTML("<em>No rows</em>")); return
    columns = columns or list(rows[0])
    shown = rows[:max_rows]
    head = "".join(f"<th>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(r.get(c, '')))}</td>" for c in columns) + "</tr>" for r in shown)
    suffix = f"<p><em>Showing {len(shown)} of {len(rows)} rows.</em></p>" if len(rows) > max_rows else ""
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>{suffix}"))

def style(fig, title, x_title=None, y_title=None, height=470, coverage_note=None):
    subtitle = f"<br><sup>{coverage_note}</sup>" if coverage_note else ""
    fig.update_layout(
        title={"text": title + subtitle, "x": 0.01, "xanchor": "left"},
        template="plotly_white", autosize=True, height=height,
        margin=dict(l=80, r=35, t=90 if coverage_note else 65, b=65),
        font=dict(family="Arial, sans-serif", size=13, color=PALETTE["charcoal"]),
        hovermode="closest", hoverlabel=dict(bgcolor="white", font_color=PALETTE["charcoal"]),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        xaxis=dict(title=x_title, gridcolor=PALETTE["grid"], zerolinecolor=PALETTE["charcoal"]),
        yaxis=dict(title=y_title, gridcolor=PALETTE["grid"], zerolinecolor=PALETTE["charcoal"]),
    )
    return fig

def show(fig):
    fig.show(config={"responsive": True, "displaylogo": False, "scrollZoom": False})

def add_decade_bands(fig, decades):
    ds = sorted(set(int(d) for d in decades if d is not None))
    for i, decade in enumerate(ds):
        if i % 2 == 0:
            fig.add_vrect(x0=decade, x1=decade + 10, fillcolor=PALETTE["light"], opacity=.42, line_width=0, layer="below")
    if ds:
        fig.update_xaxes(type="linear", tick0=ds[0], dtick=20, tickformat="d")

def coverage_note(analyzed, eligible, unit="people"):
    pct = 100 * analyzed / eligible if eligible else 0
    return f"Analyzed n={analyzed:,} of {eligible:,} eligible {unit} ({pct:.1f}% with required recorded dates)"

tables = {r[0] for r in con.execute("SHOW TABLES").fetchall()}
required = {"people", "families", "family_children", "parent_child", "partner_associations", "person_generation", "person_component"}
assert required <= tables, f"Missing normalized tables: {sorted(required - tables)}"
print(f"Database: {DB_PATH}")
print(f"Normalized tables checked: {len(required)}")

Database: /Users/johnpyrce/repos/Genealogy/artifacts/analytics/genealogy_analytics.duckdb
Normalized tables checked: 7


In [2]:
inventory = query("""
SELECT 'People' measure, COUNT(*)::INTEGER metric_value FROM people
UNION ALL SELECT 'Family records', COUNT(*)::INTEGER FROM families
UNION ALL SELECT 'Recorded two-partner unions', COUNT(DISTINCT family_id)::INTEGER FROM partner_associations
UNION ALL SELECT 'Generation levels', MAX(generation)::INTEGER FROM person_generation
UNION ALL SELECT 'Parent–child components', COUNT(DISTINCT component_id)::INTEGER FROM person_component
""")
schema_fields = {r["column_name"] for r in query("DESCRIBE people")}
assert len(query("SELECT id FROM people GROUP BY id HAVING COUNT(*) > 1")) == 0
assert query("SELECT COUNT(*) n FROM people")[0]["n"] == query("SELECT COUNT(*) n FROM person_generation")[0]["n"]
table(inventory, ["measure", "metric_value"])
print("People-table fields:", ", ".join(sorted(schema_fields)))

measure,metric_value
People,164
Family records,63
Recorded two-partner unions,60
Generation levels,9
Parent–child components,26


People-table fields: birth_year, death_year, family_name_group, first_name, id, note, source, surname


## Results

### 2. Project overview and data coverage

In [3]:
coverage = query("""
WITH parent_counts AS (
  SELECT p.id, COUNT(DISTINCT pc.parent_id) recorded_parents
  FROM people p LEFT JOIN parent_child pc ON pc.child_id=p.id GROUP BY p.id
), measures AS (
  SELECT 'Birth year' field, COUNT(birth_year)::INTEGER known, COUNT(*)::INTEGER eligible FROM people
  UNION ALL SELECT 'Death year', COUNT(death_year)::INTEGER, COUNT(*)::INTEGER FROM people
  UNION ALL SELECT 'Both dates', COUNT(*) FILTER (WHERE birth_year IS NOT NULL AND death_year IS NOT NULL)::INTEGER, COUNT(*)::INTEGER FROM people
  UNION ALL SELECT 'At least one parent', COUNT(*) FILTER (WHERE recorded_parents > 0)::INTEGER, COUNT(*)::INTEGER FROM parent_counts
  UNION ALL SELECT 'Two parents', COUNT(*) FILTER (WHERE recorded_parents = 2)::INTEGER, COUNT(*)::INTEGER FROM parent_counts
  UNION ALL SELECT 'Recorded partner', COUNT(DISTINCT person_id)::INTEGER, (SELECT COUNT(*)::INTEGER FROM people) FROM partner_associations
)
SELECT *, ROUND(100.0*known/eligible,1) coverage_pct FROM measures ORDER BY coverage_pct
""")
fig = go.Figure(go.Bar(
    x=[r['coverage_pct'] for r in coverage], y=[r['field'] for r in coverage], orientation='h',
    marker_color=PALETTE['blue'], text=[f"{r['coverage_pct']:.1f}% · {r['known']}/{r['eligible']}" for r in coverage],
    textposition='outside', customdata=[[r['known'], r['eligible']] for r in coverage],
    hovertemplate="%{y}<br>Coverage: %{x:.1f}%<br>Known: %{customdata[0]} of %{customdata[1]}<extra></extra>"
))
style(fig, "Coverage of represented fields", "Coverage (%)", None, 430)
fig.update_xaxes(range=[0, 105], ticksuffix="%", dtick=20)
show(fig)

unavailable = [
    {'requested field':'Birthplace','status':'Not represented in people or family tables'},
    {'requested field':'Residence','status':'Not represented in people or family tables'},
    {'requested field':'Sex','status':'Not represented; father/mother are relationship roles only'},
    {'requested field':'Marriage date','status':'Not represented; first-child decade is used only as an explicit proxy'},
    {'requested field':'Value-state / privacy flags','status':'Unknown, N/A, inferred, living/private are not separate fields'},
]
table(unavailable)

requested field,status
Birthplace,Not represented in people or family tables
Residence,Not represented in people or family tables
Sex,Not represented; father/mother are relationship roles only
Marriage date,Not represented; first-child decade is used only as an explicit proxy
Value-state / privacy flags,"Unknown, N/A, inferred, living/private are not separate fields"


In [4]:
parents = query("""
WITH counts AS (
 SELECT p.id, COUNT(DISTINCT pc.parent_id)::INTEGER recorded_parents
 FROM people p LEFT JOIN parent_child pc ON pc.child_id=p.id GROUP BY p.id
)
SELECT recorded_parents, COUNT(*)::INTEGER people FROM counts GROUP BY recorded_parents ORDER BY recorded_parents
""")
components = query("SELECT component_id, COUNT(*)::INTEGER people FROM person_component GROUP BY component_id ORDER BY people DESC, component_id")
fig = make_subplots(rows=1, cols=2, subplot_titles=("Recorded parent links", "Parent–child component sizes"))
fig.add_bar(x=[str(r['recorded_parents']) for r in parents], y=[r['people'] for r in parents], name='People', marker_color=PALETTE['gold'],
            text=[r['people'] for r in parents], customdata=[[r['recorded_parents']] for r in parents],
            hovertemplate="Recorded parents: %{customdata[0]}<br>People: %{y}<extra></extra>", row=1, col=1)
fig.add_bar(x=[str(r['component_id']) for r in components], y=[r['people'] for r in components], name='Component size', marker_color=PALETTE['blue'],
            hovertemplate="Component: %{x}<br>People: %{y}<extra></extra>", row=1, col=2)
style(fig, "Relationship coverage and connectivity", "Recorded parents / component ID", "People", 440)
fig.update_layout(showlegend=False)
show(fig)
print(f"Largest parent–child component: {components[0]['people']} people; isolated one-person components: {sum(r['people']==1 for r in components)}.")

Largest parent–child component: 139 people; isolated one-person components: 25.


In [5]:
heat = query("""
WITH top_groups AS (
  SELECT family_name_group FROM people WHERE family_name_group <> 'Unknown'
  GROUP BY family_name_group HAVING COUNT(*) >= 3 ORDER BY COUNT(*) DESC, family_name_group LIMIT 8
), base AS (
 SELECT p.family_name_group branch, pg.generation,
        COUNT(*)::INTEGER people,
        COUNT(p.birth_year)::INTEGER birth_known,
        COUNT(p.death_year)::INTEGER death_known,
        COUNT(*) FILTER (WHERE EXISTS (SELECT 1 FROM parent_child pc WHERE pc.child_id=p.id))::INTEGER parent_known
 FROM people p JOIN person_generation pg ON pg.person_id=p.id
 WHERE p.family_name_group IN (SELECT family_name_group FROM top_groups)
 GROUP BY p.family_name_group, pg.generation
)
SELECT branch, generation, people,
       ROUND(100.0*(birth_known + death_known + parent_known)/(3*people),1) completeness_pct,
       birth_known, death_known, parent_known
FROM base ORDER BY branch, generation
""")
branches = sorted({r['branch'] for r in heat})
generations = list(range(1, max(r['generation'] for r in heat)+1))
lookup = {(r['branch'],r['generation']):r for r in heat}
z, text = [], []
for branch in branches:
    zr, tr = [], []
    for gen in generations:
        r = lookup.get((branch,gen))
        zr.append(r['completeness_pct'] if r else None)
        tr.append((f"{branch}<br>Generation: {gen}<br>Composite coverage: {r['completeness_pct']:.1f}%"
                   f"<br>People: {r['people']}<br>Birth known: {r['birth_known']}<br>Death known: {r['death_known']}<br>Parent link: {r['parent_known']}") if r else f"{branch}<br>Generation: {gen}<br>No people")
    z.append(zr)
    text.append(tr)
fig = go.Figure(go.Heatmap(z=z, x=generations, y=branches, text=text, hovertemplate="%{text}<extra></extra>",
                           colorscale=[[0,'#F4F6F8'],[.5,'#9EB6CF'],[1,PALETTE['blue']]], zmin=0, zmax=100,
                           colorbar=dict(title='Coverage %', ticksuffix='%')))
style(fig, "Completeness heatmap by family-name branch and generation", "Generation", "Family-name group", 520,
      "Composite = mean availability of birth year, death year, and any parent link; blank cells contain no people")
fig.update_xaxes(dtick=1)
show(fig)

### 3. Tree structure

Founder and descendant metrics count only documented parent–child paths. Cross-family “bridges” below are marriage links between normalized family-name groups; this is a structural proxy, not a claim about cultural identity or biological ancestry.

In [6]:
founders = query("""
WITH RECURSIVE descent(founder_id, descendant_id) AS (
  SELECT p.id::INTEGER, p.id::INTEGER FROM people p
  WHERE NOT EXISTS (SELECT 1 FROM parent_child pc WHERE pc.child_id=p.id)
  UNION
  SELECT d.founder_id, pc.child_id FROM descent d JOIN parent_child pc ON pc.parent_id=d.descendant_id
)
SELECT f.id, TRIM(CONCAT_WS(' ',f.first_name,f.surname)) founder,
       COUNT(DISTINCT d.descendant_id)-1 AS documented_descendants
FROM descent d JOIN people f ON f.id=d.founder_id
GROUP BY f.id,f.first_name,f.surname HAVING COUNT(DISTINCT d.descendant_id)>1
ORDER BY documented_descendants DESC, founder
""")
top_founders = founders[:15]
fig = go.Figure(go.Bar(x=[r['documented_descendants'] for r in top_founders][::-1], y=[r['founder'] for r in top_founders][::-1], orientation='h',
                       marker_color=PALETTE['blue'], text=[r['documented_descendants'] for r in top_founders][::-1], textposition='outside',
                       customdata=[[r['id']] for r in top_founders][::-1],
                       hovertemplate="%{y}<br>Person ID: %{customdata[0]}<br>Documented descendants: %{x}<extra></extra>"))
style(fig, "Founders with the most documented descendants", "Documented descendants", None, 560)
show(fig)
print("All founders with at least one documented descendant:")
table(founders, ['id','founder','documented_descendants'], max_rows=100)

All founders with at least one documented descendant:


id,founder,documented_descendants
2,Teresa Bartmanowicz,72
1,Wawrzyniec Gościński,72
155,Zofia Matusiewicz,69
147,Józef Szost,68
148,Maria Matusiewicz,68
8,Maria Sasała,68
150,Zofia Wilczyńska,67
131,Małgorzata Kałucka,57
130,Wawrzyniec Miczulski,57
134,Katarzyna Fedorczak,56


In [7]:
branch_depth = query("""
SELECT p.family_name_group branch, COUNT(*)::INTEGER people,
       MIN(pg.generation)::INTEGER first_generation, MAX(pg.generation)::INTEGER last_generation,
       (MAX(pg.generation)-MIN(pg.generation)+1)::INTEGER spanned_levels
FROM people p JOIN person_generation pg ON pg.person_id=p.id
WHERE p.family_name_group <> 'Unknown'
GROUP BY p.family_name_group HAVING COUNT(*) >= 3
ORDER BY spanned_levels DESC, people DESC, branch
""")
fig = go.Figure(go.Bar(x=[r['spanned_levels'] for r in branch_depth], y=[r['branch'] for r in branch_depth], orientation='h',
                       marker_color=PALETTE['olive'], text=[f"G{r['first_generation']}–G{r['last_generation']} · n={r['people']}" for r in branch_depth],
                       textposition='inside', customdata=[[r['people'],r['first_generation'],r['last_generation']] for r in branch_depth],
                       hovertemplate="%{y}<br>Generation levels spanned: %{x}<br>People: %{customdata[0]}<br>Range: G%{customdata[1]}–G%{customdata[2]}<extra></extra>"))
style(fig, "Documented generation span by family-name group", "Generation levels spanned", None, 500)
fig.update_yaxes(autorange='reversed')
show(fig)

In [8]:
major = query("SELECT family_name_group branch FROM people GROUP BY family_name_group HAVING COUNT(*) >= 7 ORDER BY COUNT(*) DESC")
major_names = [r['branch'] for r in major]
placeholders = ','.join('?' for _ in major_names)
branch_time = query(f"""
SELECT family_name_group AS branch, (FLOOR(birth_year/10)*10)::INTEGER AS decade, COUNT(*)::INTEGER AS births
FROM people WHERE birth_year IS NOT NULL AND family_name_group IN ({placeholders})
GROUP BY family_name_group, decade ORDER BY decade, branch
""", major_names)
known_births = query("SELECT COUNT(birth_year)::INTEGER n, COUNT(*)::INTEGER total FROM people")[0]
fig = go.Figure()
colors = [PALETTE['blue'],PALETTE['gold'],PALETTE['orange'],PALETTE['olive'],PALETTE['pink']]
for i, branch in enumerate(major_names):
    rows = [r for r in branch_time if r['branch']==branch]
    fig.add_scatter(x=[r['decade'] for r in rows], y=[r['births'] for r in rows], mode='lines+markers+text', name=branch,
                    line=dict(color=colors[i%len(colors)], width=2), marker=dict(size=8),
                    text=[branch]+['']*(len(rows)-1), textposition='middle left',
                    hovertemplate=f"{branch}<br>Birth decade: %{{x:.0f}}s<br>Recorded births: %{{y:.0f}}<extra></extra>")
add_decade_bands(fig, [r['decade'] for r in branch_time])
style(fig, "Recorded branch sizes over time", "Birth decade", "People with recorded birth", 500,
      coverage_note(known_births['n'],known_births['total']))
show(fig)

In [9]:
role_counts = query("SELECT branch_role, COUNT(*)::INTEGER family_records FROM families GROUP BY branch_role ORDER BY family_records DESC")
sibling_groups = query("""
SELECT f.id family_id, TRIM(CONCAT_WS(' ',fa.first_name,fa.surname)) father,
       TRIM(CONCAT_WS(' ',mo.first_name,mo.surname)) mother, COUNT(fc.child_id)::INTEGER recorded_children
FROM families f LEFT JOIN people fa ON fa.id=f.father_id LEFT JOIN people mo ON mo.id=f.mother_id
JOIN family_children fc ON fc.family_id=f.id GROUP BY f.id,fa.first_name,fa.surname,mo.first_name,mo.surname
ORDER BY recorded_children DESC,family_id LIMIT 12
""")
fig = make_subplots(rows=1, cols=2, subplot_titles=("Family records by branch role", "Largest recorded sibling groups"))
fig.add_bar(x=[r['branch_role'] for r in role_counts], y=[r['family_records'] for r in role_counts], marker_color=PALETTE['gold'],
            name='Family records', hovertemplate="Role: %{x}<br>Family records: %{y}<extra></extra>", row=1,col=1)
labels=[f"F{r['family_id']} · {r['father']} + {r['mother']}" for r in sibling_groups]
fig.add_bar(x=labels, y=[r['recorded_children'] for r in sibling_groups], marker_color=PALETTE['blue'], name='Recorded children',
            hovertemplate="%{x}<br>Recorded children: %{y}<extra></extra>", row=1,col=2)
style(fig,"Recorded line roles and sibling groups",None,"Count",520)
fig.update_xaxes(tickangle=-45,row=1,col=2)
fig.update_layout(showlegend=False)
show(fig)
print("Branch role is a family-record classification. A person-level direct-ancestor count requires a designated focal person, which the schema does not provide.")

Branch role is a family-record classification. A person-level direct-ancestor count requires a designated focal person, which the schema does not provide.


In [10]:
terminal = query("""
SELECT p.family_name_group AS branch,
       COUNT(*) FILTER (WHERE EXISTS (SELECT 1 FROM parent_child pc WHERE pc.parent_id=p.id))::INTEGER AS continuing_in_record,
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM parent_child pc WHERE pc.parent_id=p.id))::INTEGER AS terminal_in_record,
       COUNT(*)::INTEGER AS people
FROM people p WHERE p.family_name_group <> 'Unknown'
GROUP BY p.family_name_group HAVING COUNT(*) >= 3 ORDER BY people DESC,branch
""")
fig=go.Figure()
fig.add_bar(x=[r['branch'] for r in terminal],y=[r['continuing_in_record'] for r in terminal],name='Has recorded child link',
            marker_color=PALETTE['blue'],customdata=[[r['people']] for r in terminal],
            hovertemplate="%{x}<br>People with recorded child link: %{y}<br>Group total: %{customdata[0]}<extra></extra>")
fig.add_bar(x=[r['branch'] for r in terminal],y=[r['terminal_in_record'] for r in terminal],name='No recorded child link',
            marker_color=PALETTE['light'],marker_line_color=PALETTE['charcoal'],marker_line_width=1,
            customdata=[[r['people']] for r in terminal],
            hovertemplate="%{x}<br>People without recorded child link: %{y}<br>Group total: %{customdata[0]}<extra></extra>")
style(fig,"Continuing and terminal lines in the record","Family-name group","People",500,
      "Terminal means no documented child link; it does not establish biological extinction")
fig.update_layout(barmode='stack')
fig.update_xaxes(tickangle=-45)
show(fig)

In [11]:
cross_unions = query("""
SELECT LEAST(a.family_name_group,b.family_name_group) branch_a,
       GREATEST(a.family_name_group,b.family_name_group) branch_b,
       COUNT(DISTINCT pa.family_id)::INTEGER unions
FROM partner_associations pa JOIN people a ON a.id=pa.person_id JOIN people b ON b.id=pa.partner_id
WHERE a.family_name_group<>b.family_name_group AND pa.person_id<pa.partner_id
GROUP BY branch_a,branch_b ORDER BY unions DESC,branch_a,branch_b LIMIT 15
""")
bridge_people = query("""
SELECT p.id, TRIM(CONCAT_WS(' ',p.first_name,p.surname)) person,
       COUNT(DISTINCT partner.family_name_group)::INTEGER partner_branches,
       STRING_AGG(DISTINCT partner.family_name_group, ', ' ORDER BY partner.family_name_group) connected_groups
FROM people p JOIN partner_associations pa ON pa.person_id=p.id JOIN people partner ON partner.id=pa.partner_id
WHERE partner.family_name_group<>p.family_name_group
GROUP BY p.id,p.first_name,p.surname HAVING COUNT(DISTINCT partner.family_name_group)>0
ORDER BY partner_branches DESC,person LIMIT 15
""")
fig = go.Figure(go.Bar(x=[r['unions'] for r in cross_unions][::-1], y=[f"{r['branch_a']} ↔ {r['branch_b']}" for r in cross_unions][::-1], orientation='h',
                       marker_color=PALETTE['orange'], text=[r['unions'] for r in cross_unions][::-1], textposition='outside',
                       hovertemplate="%{y}<br>Recorded unions: %{x}<extra></extra>"))
style(fig,"Recorded unions linking family-name groups","Recorded unions",None,520)
show(fig)
print("Potential bridge people (cross-group partner links):")
table(bridge_people,['id','person','partner_branches','connected_groups'],15)

Potential bridge people (cross-group partner links):


id,person,partner_branches,connected_groups
6,Antoni Gościński,3,"Grotkowski, Sasała, Tryszczyła"
163,Jan Miczulski,2,"Bukowski, Gruczelak"
123,Michał Rams,2,"Fedorczak, Unknown"
91,Agata,1,Rams
141,Agnieszka Homa,1,Miczulski
71,Agnieszka Pluta,1,Wiklowski
52,Andrzej Drabyk,1,Guzyk
77,Anna Bukowska,1,Pyrc
113,Anna Kokoszka,1,Romer
29,Anna Maślanka,1,Gościński


In [12]:
# Shared-ancestor partner pairs are a conservative structural flag for cousin/kin marriages.
edges = query("SELECT DISTINCT parent_id, child_id FROM parent_child")
parents_of = defaultdict(set)
for e in edges: parents_of[e['child_id']].add(e['parent_id'])
def ancestors(person_id):
    seen=set(); queue=deque(parents_of.get(person_id,set()))
    while queue:
        x=queue.popleft()
        if x in seen: continue
        seen.add(x); queue.extend(parents_of.get(x,set())-seen)
    return seen
people_rows = query("SELECT id, TRIM(CONCAT_WS(' ',first_name,surname)) AS person_name FROM people")
names = {r['id']:r['person_name'] for r in people_rows}
pairs = query("SELECT family_id,person_id,partner_id FROM partner_associations WHERE person_id<partner_id ORDER BY family_id")
kin_flags=[]
for pair in pairs:
    shared=ancestors(pair['person_id']) & ancestors(pair['partner_id'])
    if shared:
        kin_flags.append({'family_id':pair['family_id'],'partners':f"{names[pair['person_id']]} + {names[pair['partner_id']]}",
                          'shared_ancestor_count':len(shared),'shared_ancestors':', '.join(names[x] for x in sorted(shared))})
print(f"Recorded two-partner unions checked: {len(pairs)}; unions with at least one documented shared ancestor: {len(kin_flags)}")
table(kin_flags, max_rows=20)

Recorded two-partner unions checked: 60; unions with at least one documented shared ancestor: 0


### 4. Timeline and generational patterns

No marriage dates are recorded. The “family formation” series uses the first recorded child’s birth decade and is visibly labeled as a proxy. Counts of people alive require both birth and death years; missing death dates are excluded, never treated as still living.

In [13]:
events = query("""
WITH decades AS (
 SELECT decade FROM range(1760,2031,10) t(decade)
), births AS (SELECT (FLOOR(birth_year/10)*10)::INTEGER AS decade,COUNT(*)::INTEGER AS n FROM people WHERE birth_year IS NOT NULL GROUP BY decade),
deaths AS (SELECT (FLOOR(death_year/10)*10)::INTEGER AS decade,COUNT(*)::INTEGER AS n FROM people WHERE death_year IS NOT NULL GROUP BY decade),
formation AS (
 SELECT (FLOOR(MIN(p.birth_year)/10)*10)::INTEGER AS decade,COUNT(DISTINCT f.id)::INTEGER AS n
 FROM families f JOIN family_children fc ON fc.family_id=f.id JOIN people p ON p.id=fc.child_id
 WHERE p.birth_year IS NOT NULL GROUP BY f.id
), formation_by_decade AS (SELECT decade,COUNT(*)::INTEGER n FROM formation GROUP BY decade)
SELECT d.decade,COALESCE(b.n,0) births,COALESCE(x.n,0) deaths,COALESCE(f.n,0) family_formation_proxy
FROM decades d LEFT JOIN births b USING(decade) LEFT JOIN deaths x USING(decade) LEFT JOIN formation_by_decade f USING(decade)
ORDER BY decade
""")
fig=go.Figure()
for field,label,color,dash in [('births','Births',PALETTE['blue'],'solid'),('deaths','Deaths',PALETTE['orange'],'dash'),('family_formation_proxy','Family formation proxy',PALETTE['olive'],'dot')]:
    fig.add_scatter(x=[r['decade'] for r in events],y=[r[field] for r in events],mode='lines+markers+text',name=label,
                    line=dict(color=color,dash=dash,width=2),marker=dict(size=7),
                    text=[label]+['']*(len(events)-1),textposition='middle left',
                    hovertemplate=f"{label}<br>Decade: %{{x:.0f}}s<br>Count: %{{y:.0f}}<extra></extra>")
add_decade_bands(fig,[r['decade'] for r in events])
style(fig,"Recorded events by decade","Decade","Recorded events",500,
      "Births: 77/164 people (47.0%); deaths: 42/164 (25.6%); formation proxy: families with ≥1 dated child")
show(fig)

In [14]:
alive = query("""
WITH decades AS (SELECT decade FROM range(1760,2031,10) t(decade)), dated AS (
 SELECT * FROM people WHERE birth_year IS NOT NULL AND death_year IS NOT NULL AND death_year>=birth_year
)
SELECT decade, COUNT(*) FILTER (WHERE birth_year<=decade+9 AND death_year>=decade)::INTEGER people_alive_during_decade
FROM decades CROSS JOIN dated GROUP BY decade ORDER BY decade
""")
dated_n=query("SELECT COUNT(*)::INTEGER n FROM people WHERE birth_year IS NOT NULL AND death_year IS NOT NULL AND death_year>=birth_year")[0]['n']
total=query("SELECT COUNT(*)::INTEGER n FROM people")[0]['n']
fig=go.Figure(go.Scatter(x=[r['decade'] for r in alive],y=[r['people_alive_during_decade'] for r in alive],mode='lines+markers+text',
                         name='People alive',line=dict(color=PALETTE['blue'],width=3),marker=dict(size=8),
                         text=['People alive']+['']*(len(alive)-1),textposition='middle left',
                         hovertemplate="Decade: %{x:.0f}s<br>People alive during decade: %{y:.0f}<extra></extra>"))
add_decade_bands(fig,[r['decade'] for r in alive])
style(fig,"People known to be alive during each decade","Decade","People",470,coverage_note(dated_n,total))
show(fig)

In [15]:
parent_ages=query("""
SELECT pc.parent_role, child.birth_year-parent.birth_year age,
       TRIM(CONCAT_WS(' ',parent.first_name,parent.surname)) parent,
       TRIM(CONCAT_WS(' ',child.first_name,child.surname)) child
FROM parent_child pc JOIN people parent ON parent.id=pc.parent_id JOIN people child ON child.id=pc.child_id
WHERE parent.birth_year IS NOT NULL AND child.birth_year IS NOT NULL AND child.birth_year>=parent.birth_year
""")
eligible_links=query("SELECT COUNT(*)::INTEGER n FROM parent_child")[0]['n']
fig=go.Figure()
for role,color in [('father',PALETTE['blue']),('mother',PALETTE['gold'])]:
    rows=[r for r in parent_ages if r['parent_role']==role]
    fig.add_histogram(x=[r['age'] for r in rows],name=role.title(),marker_color=color,opacity=.72,xbins=dict(start=10,end=61,size=5),
                      customdata=[[r['parent'],r['child']] for r in rows],
                      hovertemplate=f"{role.title()} age band: %{{x}}<br>Observations: %{{y}}<extra></extra>")
style(fig,"Recorded parental age at a child’s birth","Parent age (years)","Parent–child observations",470,
      coverage_note(len(parent_ages),eligible_links,"parent–child links"))
fig.update_layout(barmode='overlay')
show(fig)
ages=[r['age'] for r in parent_ages]
print(f"Typical recorded generation length: median {sorted(ages)[len(ages)//2]} years; mean {sum(ages)/len(ages):.1f} years (n={len(ages)} dated links).")

Typical recorded generation length: median 32 years; mean 33.1 years (n=79 dated links).


In [16]:
spouse_ages=query("""
SELECT pa.family_id, ABS(a.birth_year-b.birth_year)::INTEGER age_gap,
       TRIM(CONCAT_WS(' ',a.first_name,a.surname)) partner_1,
       TRIM(CONCAT_WS(' ',b.first_name,b.surname)) partner_2
FROM partner_associations pa JOIN people a ON a.id=pa.person_id JOIN people b ON b.id=pa.partner_id
WHERE pa.person_id<pa.partner_id AND a.birth_year IS NOT NULL AND b.birth_year IS NOT NULL
ORDER BY age_gap DESC
""")
unions=query("SELECT COUNT(DISTINCT family_id)::INTEGER n FROM partner_associations")[0]['n']
fig=go.Figure(go.Histogram(x=[r['age_gap'] for r in spouse_ages],xbins=dict(start=0,end=31,size=2),marker_color=PALETTE['pink'],
                           hovertemplate="Absolute age gap: %{x} years<br>Couples: %{y}<extra></extra>"))
style(fig,"Absolute recorded age differences between partners","Absolute age gap (years)","Recorded couples",430,
      coverage_note(len(spouse_ages),unions,"recorded unions"))
show(fig)

overlap=query("""
WITH grand_links AS (
 SELECT DISTINCT gp.parent_id grandparent_id, pc.child_id grandchild_id
 FROM parent_child gp JOIN parent_child pc ON pc.parent_id=gp.child_id
), dated AS (
 SELECT g.*,gp.death_year,gc.birth_year
 FROM grand_links g JOIN people gp ON gp.id=g.grandparent_id JOIN people gc ON gc.id=g.grandchild_id
 WHERE gp.death_year IS NOT NULL AND gc.birth_year IS NOT NULL
)
SELECT COUNT(*)::INTEGER dated_pairs,
       COUNT(*) FILTER (WHERE death_year>=birth_year)::INTEGER alive_at_birth,
       COUNT(*) FILTER (WHERE death_year>=birth_year+12)::INTEGER lived_through_age_12
FROM dated
""")[0]
grand_total=query("WITH g AS (SELECT DISTINCT gp.parent_id a,pc.child_id b FROM parent_child gp JOIN parent_child pc ON pc.parent_id=gp.child_id) SELECT COUNT(*)::INTEGER n FROM g")[0]['n']
table([{'measure':'Dated grandparent–grandchild pairs','value':overlap['dated_pairs'],'coverage':f"{100*overlap['dated_pairs']/grand_total:.1f}% of {grand_total}"},
       {'measure':'Grandparent recorded alive at grandchild birth','value':overlap['alive_at_birth'],'coverage':f"{100*overlap['alive_at_birth']/overlap['dated_pairs']:.1f}% of dated pairs"},
       {'measure':'Grandparent recorded alive through grandchild age 12','value':overlap['lived_through_age_12'],'coverage':f"{100*overlap['lived_through_age_12']/overlap['dated_pairs']:.1f}% of dated pairs"}])

measure,value,coverage
Dated grandparent–grandchild pairs,56,24.9% of 225
Grandparent recorded alive at grandchild birth,37,66.1% of dated pairs
Grandparent recorded alive through grandchild age 12,23,41.1% of dated pairs


### 5. Longevity and mortality

The charts use only people with both dates and valid nonnegative lifespans. Cohort comparisons are limited to birth years through 1925. Sex comparisons are unavailable because sex is not an analytical field; father/mother roles cannot safely be generalized to all people.

In [17]:
lifespans=query("""
SELECT id,TRIM(CONCAT_WS(' ',first_name,surname)) person,family_name_group branch,birth_year,death_year,
       (death_year-birth_year)::INTEGER lifespan
FROM people WHERE birth_year IS NOT NULL AND death_year IS NOT NULL AND death_year>=birth_year
""")
fig=go.Figure(go.Histogram(x=[r['lifespan'] for r in lifespans],xbins=dict(start=0,end=111,size=10),marker_color=PALETTE['blue'],
                           customdata=[[r['person'],r['birth_year'],r['death_year']] for r in lifespans],
                           hovertemplate="Age band: %{x}<br>People: %{y}<extra></extra>"))
style(fig,"Age at death from recorded birth and death years","Age at death (years)","People",430,coverage_note(len(lifespans),total))
show(fig)

In [18]:
cohorts=query("""
SELECT (FLOOR(birth_year/20)*20)::INTEGER AS cohort_start, COUNT(*)::INTEGER AS people,
       MEDIAN(death_year-birth_year)::DOUBLE median_lifespan
FROM people WHERE birth_year IS NOT NULL AND death_year IS NOT NULL AND death_year>=birth_year AND birth_year<=1925
GROUP BY cohort_start ORDER BY cohort_start
""")
mature_births=query("SELECT COUNT(*)::INTEGER n FROM people WHERE birth_year IS NOT NULL AND birth_year<=1925")[0]['n']
fig=go.Figure(go.Scatter(x=[r['cohort_start'] for r in cohorts],y=[r['median_lifespan'] for r in cohorts],mode='lines+markers+text',
                         name='Median lifespan',line=dict(color=PALETTE['blue'],width=3),marker=dict(size=9),
                         text=['Median lifespan']+['']*(len(cohorts)-1),textposition='middle left',
                         customdata=[[r['people']] for r in cohorts],
                         hovertemplate="Birth cohort: %{x:.0f}–%{customdata[0]} observations<br>Median lifespan: %{y:.0f} years<extra></extra>"))
add_decade_bands(fig,[r['cohort_start'] for r in cohorts])
style(fig,"Median recorded lifespan by 20-year birth cohort","Birth cohort start","Median lifespan (years)",450,
      coverage_note(sum(r['people'] for r in cohorts),mature_births,"people born through 1925"))
fig.update_xaxes(dtick=20)
show(fig)

In [19]:
branch_life=query("""
SELECT family_name_group branch,COUNT(*)::INTEGER people,MEDIAN(death_year-birth_year)::DOUBLE median_lifespan
FROM people WHERE birth_year IS NOT NULL AND death_year IS NOT NULL AND death_year>=birth_year AND birth_year<=1925
GROUP BY family_name_group HAVING COUNT(*)>=2 ORDER BY median_lifespan DESC,people DESC
""")
fig=go.Figure(go.Bar(x=[r['median_lifespan'] for r in branch_life],y=[r['branch'] for r in branch_life],orientation='h',
                     marker_color=PALETTE['olive'],text=[f"{r['median_lifespan']:.0f} y · n={r['people']}" for r in branch_life],textposition='outside',
                     customdata=[[r['people']] for r in branch_life],
                     hovertemplate="%{y}<br>Median lifespan: %{x:.0f} years<br>Analyzed people: %{customdata[0]}<extra></extra>"))
style(fig,"Recorded lifespan by family-name group (mature cohorts)","Median lifespan (years)",None,430,
      "Only groups with n≥2 dated deaths among births through 1925; small samples are descriptive")
fig.update_yaxes(autorange='reversed')
show(fig)

In [20]:
mortality=[{'band':'Infant (<1)','people':sum(r['lifespan']<1 for r in lifespans)},
           {'band':'Child (1–4)','people':sum(1<=r['lifespan']<=4 for r in lifespans)},
           {'band':'Child (5–14)','people':sum(5<=r['lifespan']<=14 for r in lifespans)},
           {'band':'15+','people':sum(r['lifespan']>=15 for r in lifespans)}]
fig=go.Figure(go.Bar(x=[r['band'] for r in mortality],y=[r['people'] for r in mortality],marker_color=[PALETTE['orange']]*3+[PALETTE['blue']],
                     text=[r['people'] for r in mortality],textposition='outside',hovertemplate="Age band: %{x}<br>People: %{y}<extra></extra>"))
style(fig,"Recorded mortality age bands","Age at death band","People",410,coverage_note(len(lifespans),total))
show(fig)

long_lived=sorted(lifespans,key=lambda r:(-r['lifespan'],r['person']))[:12]
table(long_lived,['id','person','branch','birth_year','death_year','lifespan'],12)

id,person,branch,birth_year,death_year,lifespan
128,Henryk Rams,Rams,1927,2024,97
83,Bronisława Pyrc,Pyrc,1913,2009,96
118,Helena Ruchałowska,Ruchałowski,1909,2005,96
84,Stefania Pyrc,Pyrc,1923,2018,95
152,Piotr Jacenik,Jacenik,1908,2002,94
94,Joanna Gruczelak,Gruczelak,1901,1994,93
137,Jan Moszczak,Moszczak,1860,1952,92
138,Maria Wilczyńska,Wilczyński,1861,1953,92
146,Maria Moszczak,Moszczak,1903,1993,90
126,Stanisława Rams,Rams,1923,2013,90


In [21]:
couple_life=query("""
SELECT pa.family_id,TRIM(CONCAT_WS(' ',a.first_name,a.surname)) partner_1,
       TRIM(CONCAT_WS(' ',b.first_name,b.surname)) partner_2,
       (a.death_year-a.birth_year)::INTEGER lifespan_1,(b.death_year-b.birth_year)::INTEGER lifespan_2
FROM partner_associations pa JOIN people a ON a.id=pa.person_id JOIN people b ON b.id=pa.partner_id
WHERE pa.person_id<pa.partner_id AND a.birth_year IS NOT NULL AND a.death_year IS NOT NULL
  AND b.birth_year IS NOT NULL AND b.death_year IS NOT NULL
  AND a.death_year>=a.birth_year AND b.death_year>=b.birth_year
""")
fig=go.Figure(go.Scatter(x=[r['lifespan_1'] for r in couple_life],y=[r['lifespan_2'] for r in couple_life],mode='markers',
                         marker=dict(size=10,color=PALETTE['blue'],line=dict(color=PALETTE['charcoal'],width=1)),
                         customdata=[[r['family_id'],r['partner_1'],r['partner_2']] for r in couple_life],
                         hovertemplate="Family %{customdata[0]}<br>%{customdata[1]}: %{x:.0f} years<br>%{customdata[2]}: %{y:.0f} years<extra></extra>"))
fig.add_shape(type='line',x0=0,y0=0,x1=110,y1=110,line=dict(color=PALETTE['muted'],dash='dash'))
style(fig,"Comparative recorded longevity within couples","Partner 1 lifespan (years)","Partner 2 lifespan (years)",450,
      coverage_note(len(couple_life),unions,"recorded unions"))
fig.update_xaxes(range=[0,110],dtick=20); fig.update_yaxes(range=[0,110],dtick=20)
show(fig)

### 6. Fertility and household formation

All measures are about **recorded children**. The reasonably-complete subset improves date analysis but cannot prove that every historical child was captured.

In [22]:
family_sizes=query("""
SELECT f.id family_id,f.branch_role,COUNT(fc.child_id)::INTEGER recorded_children
FROM families f LEFT JOIN family_children fc ON fc.family_id=f.id GROUP BY f.id,f.branch_role
""")
size_dist=[]
for n in sorted({r['recorded_children'] for r in family_sizes}):
    size_dist.append({'recorded_children':n,'families':sum(r['recorded_children']==n for r in family_sizes)})
fig=go.Figure(go.Bar(x=[r['recorded_children'] for r in size_dist],y=[r['families'] for r in size_dist],marker_color=PALETTE['blue'],
                     text=[r['families'] for r in size_dist],textposition='outside',
                     hovertemplate="Recorded children: %{x}<br>Family records: %{y}<extra></extra>"))
style(fig,"Distribution of recorded children per family record","Recorded children","Family records",430)
fig.update_xaxes(dtick=1)
show(fig)

In [23]:
complete_families=query("""
WITH counts AS (
 SELECT f.id family_id,f.mother_id,COUNT(fc.child_id)::INTEGER recorded_children,
        COUNT(p.birth_year)::INTEGER dated_children,MIN(p.birth_year) first_birth,MAX(p.birth_year) last_birth
 FROM families f JOIN family_children fc ON fc.family_id=f.id JOIN people p ON p.id=fc.child_id
 GROUP BY f.id,f.mother_id
)
SELECT c.*,m.birth_year mother_birth_year,
       first_birth-m.birth_year mother_age_first,last_birth-m.birth_year mother_age_last
FROM counts c LEFT JOIN people m ON m.id=c.mother_id
WHERE recorded_children>=2 AND recorded_children=dated_children
ORDER BY recorded_children DESC,family_id
""")
intervals=query("""
WITH complete AS (
 SELECT fc.family_id,COUNT(*) n,COUNT(p.birth_year) dated
 FROM family_children fc JOIN people p ON p.id=fc.child_id GROUP BY fc.family_id HAVING COUNT(*)>=2 AND COUNT(*)=COUNT(p.birth_year)
), ordered AS (
 SELECT fc.family_id,p.birth_year,LAG(p.birth_year) OVER(PARTITION BY fc.family_id ORDER BY p.birth_year,p.id) prior_birth
 FROM family_children fc JOIN complete c ON c.family_id=fc.family_id JOIN people p ON p.id=fc.child_id
)
SELECT family_id,(birth_year-prior_birth)::INTEGER interval_years FROM ordered WHERE prior_birth IS NOT NULL
""")
fig=make_subplots(rows=1,cols=2,subplot_titles=("Recorded children in analyzable families","Intervals between recorded sibling births"))
fig.add_histogram(x=[r['recorded_children'] for r in complete_families],xbins=dict(start=1.5,end=10.5,size=1),marker_color=PALETTE['blue'],
                  name='Families',hovertemplate="Recorded children: %{x}<br>Families: %{y}<extra></extra>",row=1,col=1)
fig.add_histogram(x=[r['interval_years'] for r in intervals],xbins=dict(start=-.5,end=11.5,size=1),marker_color=PALETTE['gold'],
                  name='Sibling intervals',hovertemplate="Interval: %{x} years<br>Intervals: %{y}<extra></extra>",row=1,col=2)
style(fig,"Recorded family size and birth spacing in the analyzable subset","Years / recorded children","Count",450,
      f"Reasonably-complete child-date subset: n={len(complete_families)} of 63 family records; intervals n={len(intervals)}")
fig.update_layout(showlegend=False)
show(fig)

maternal=[r for r in complete_families if r['mother_age_first'] is not None and 10<=r['mother_age_first']<=60 and 10<=r['mother_age_last']<=60]
print(f"Maternal-role age at first recorded child: n={len(maternal)}, median={sorted(r['mother_age_first'] for r in maternal)[len(maternal)//2] if maternal else 'n/a'} years")
print(f"Maternal-role age at last recorded child: n={len(maternal)}, median={sorted(r['mother_age_last'] for r in maternal)[len(maternal)//2] if maternal else 'n/a'} years")

Maternal-role age at first recorded child: n=6, median=28 years
Maternal-role age at last recorded child: n=6, median=35 years


In [24]:
family_period=query("""
WITH family_summary AS (
 SELECT f.id,f.branch_role,COUNT(fc.child_id)::INTEGER recorded_children,
        (FLOOR(MIN(p.birth_year)/10)*10)::INTEGER AS first_child_decade
 FROM families f LEFT JOIN family_children fc ON fc.family_id=f.id LEFT JOIN people p ON p.id=fc.child_id
 GROUP BY f.id,f.branch_role
)
SELECT branch_role,first_child_decade,COUNT(*)::INTEGER families,ROUND(AVG(recorded_children),2) avg_recorded_children
FROM family_summary WHERE first_child_decade IS NOT NULL GROUP BY branch_role,first_child_decade ORDER BY first_child_decade,branch_role
""")
dated_family_count=query("SELECT COUNT(DISTINCT fc.family_id)::INTEGER n FROM family_children fc JOIN people p ON p.id=fc.child_id WHERE p.birth_year IS NOT NULL")[0]['n']
fig=go.Figure()
role_colors={'main_line':PALETTE['blue'],'spouse_ancestry':PALETTE['gold'],'collateral':PALETTE['orange']}
for role in role_colors:
    rows=[r for r in family_period if r['branch_role']==role]
    fig.add_scatter(x=[r['first_child_decade'] for r in rows],y=[r['avg_recorded_children'] for r in rows],mode='lines+markers+text',name=role,
                    line=dict(color=role_colors[role],width=2),marker=dict(size=8),text=[role]+['']*(len(rows)-1),textposition='middle left',
                    customdata=[[r['families']] for r in rows],
                    hovertemplate=f"{role}<br>First-child decade: %{{x:.0f}}s<br>Mean recorded children: %{{y:.2f}}<br>Family records: %{{customdata[0]}}<extra></extra>")
add_decade_bands(fig,[r['first_child_decade'] for r in family_period])
style(fig,"Mean recorded children by first-child decade and branch role","First recorded child decade","Mean recorded children",480,
      coverage_note(dated_family_count,63,"family records"))
show(fig)

In [25]:
households=query("""
WITH eligible AS (SELECT * FROM people WHERE birth_year IS NOT NULL AND birth_year<=2008), flags AS (
 SELECT e.id,
  EXISTS(SELECT 1 FROM partner_associations pa WHERE pa.person_id=e.id) has_partner,
  EXISTS(SELECT 1 FROM parent_child pc WHERE pc.parent_id=e.id) has_child
 FROM eligible e
)
SELECT COUNT(*)::INTEGER eligible_records,
 COUNT(*) FILTER(WHERE has_partner)::INTEGER with_partner,
 COUNT(*) FILTER(WHERE has_child)::INTEGER with_child,
 COUNT(*) FILTER(WHERE has_partner OR has_child)::INTEGER with_partner_or_child
FROM flags
""")[0]
remarriage=query("""
WITH unions AS (SELECT person_id,COUNT(DISTINCT family_id)::INTEGER unions FROM partner_associations GROUP BY person_id)
SELECT COUNT(*) FILTER(WHERE unions>1)::INTEGER people_multiple_unions,MAX(unions)::INTEGER maximum_unions FROM unions
""")[0]
blended=query("""
WITH x AS (SELECT parent_id,COUNT(DISTINCT family_id)::INTEGER child_families FROM parent_child GROUP BY parent_id)
SELECT COUNT(*) FILTER(WHERE child_families>1)::INTEGER parents_with_children_in_multiple_families,MAX(child_families)::INTEGER maximum_child_families FROM x
""")[0]
table([
 {'measure':'Birth-dated records born by 2008','value':households['eligible_records'],'note':'Age-18 eligibility proxy; not a living-population denominator'},
 {'measure':'With recorded partner','value':households['with_partner'],'note':f"{100*households['with_partner']/households['eligible_records']:.1f}% of proxy-eligible records"},
 {'measure':'With recorded child','value':households['with_child'],'note':f"{100*households['with_child']/households['eligible_records']:.1f}% of proxy-eligible records"},
 {'measure':'With partner or child','value':households['with_partner_or_child'],'note':f"{100*households['with_partner_or_child']/households['eligible_records']:.1f}% of proxy-eligible records"},
 {'measure':'People in multiple recorded unions','value':remarriage['people_multiple_unions'],'note':f"Maximum {remarriage['maximum_unions']} unions"},
 {'measure':'Parents with children in multiple family records','value':blended['parents_with_children_in_multiple_families'],'note':f"Maximum {blended['maximum_child_families']} child-bearing family records"},
])

measure,value,note
Birth-dated records born by 2008,75,Age-18 eligibility proxy; not a living-population denominator
With recorded partner,58,77.3% of proxy-eligible records
With recorded child,45,60.0% of proxy-eligible records
With partner or child,59,78.7% of proxy-eligible records
People in multiple recorded unions,3,Maximum 3 unions
Parents with children in multiple family records,0,Maximum 1 child-bearing family records


### 7. Surnames and naming patterns

Name matching uses case-insensitive exact recorded given names. It does not equate spelling variants, diminutives, or translations unless the registry already normalized them.

In [26]:
surnames=query("SELECT family_name_group,COUNT(*)::INTEGER people FROM people GROUP BY family_name_group ORDER BY people DESC,family_name_group LIMIT 20")
fig=go.Figure(go.Bar(x=[r['people'] for r in surnames][::-1],y=[r['family_name_group'] for r in surnames][::-1],orientation='h',
                     marker_color=PALETTE['blue'],text=[r['people'] for r in surnames][::-1],textposition='outside',
                     hovertemplate="Family-name group: %{y}<br>People: %{x}<extra></extra>"))
style(fig,"Most frequent normalized family-name groups","People",None,560)
show(fig)

variants=query("""
SELECT family_name_group,COUNT(DISTINCT surname)::INTEGER recorded_variants,
       STRING_AGG(DISTINCT surname, ', ' ORDER BY surname) variants
FROM people GROUP BY family_name_group HAVING COUNT(DISTINCT surname)>1 ORDER BY recorded_variants DESC,family_name_group
""")
print("Recorded surname variants within normalized groups:")
table(variants)

Recorded surname variants within normalized groups:


family_name_group,recorded_variants,variants
Drzązgowski,2,"Drzązgowska, Drzązgowski"
Gościński,2,"Gościńska, Gościński"
Miczulski,2,"Miczulska, Miczulski"
Wiklowski,2,"Wiklowska, Wiklowski"
Wiśniewski,2,"Wiśniewska, Wiśniewski"


In [27]:
first_names=query("SELECT first_name,COUNT(*)::INTEGER people FROM people WHERE first_name<>'' GROUP BY first_name ORDER BY people DESC,first_name LIMIT 15")
fig=go.Figure(go.Bar(x=[r['people'] for r in first_names][::-1],y=[r['first_name'] for r in first_names][::-1],orientation='h',
                     marker_color=PALETTE['gold'],text=[r['people'] for r in first_names][::-1],textposition='outside',
                     hovertemplate="Given name: %{y}<br>People: %{x}<extra></extra>"))
style(fig,"Most frequent recorded given names","People",None,500)
show(fig)

namesakes=query("""
WITH direct AS (
 SELECT 'Parent' relationship,COUNT(*)::INTEGER matches,COUNT(DISTINCT pc.child_id)::INTEGER named_people
 FROM parent_child pc JOIN people p ON p.id=pc.parent_id JOIN people c ON c.id=pc.child_id
 WHERE LOWER(TRIM(p.first_name))=LOWER(TRIM(c.first_name)) AND TRIM(c.first_name)<>''
), grand AS (
 SELECT 'Grandparent',COUNT(*)::INTEGER,COUNT(DISTINCT gc.id)::INTEGER
 FROM parent_child a JOIN parent_child b ON b.parent_id=a.child_id
 JOIN people gp ON gp.id=a.parent_id JOIN people gc ON gc.id=b.child_id
 WHERE LOWER(TRIM(gp.first_name))=LOWER(TRIM(gc.first_name)) AND TRIM(gc.first_name)<>''
)
SELECT * FROM direct UNION ALL SELECT * FROM grand
""")
fig=go.Figure(go.Bar(x=[r['relationship'] for r in namesakes],y=[r['matches'] for r in namesakes],marker_color=[PALETTE['blue'],PALETTE['olive']],
                     text=[f"{r['matches']} matches · {r['named_people']} people" for r in namesakes],textposition='outside',
                     customdata=[[r['named_people']] for r in namesakes],
                     hovertemplate="Relationship: %{x}<br>Exact-name links: %{y}<br>Distinct named people: %{customdata[0]}<extra></extra>"))
style(fig,"Exact given-name transmission across generations","Relationship to earlier namesake","Exact matching links",410)
show(fig)

## Takeaways

1. **Coverage, not population history, is the dominant signal.** Birth years are present for 47.0% of people and both dates for 25.6%, so demographic charts are descriptive views of dated records, not population estimates.
2. **The tree is structurally concentrated.** A 139-person parent–child component contains most records, while 25 people are isolated from parent–child links. Those isolates may still have spouse associations; component counts intentionally exclude spouses.
3. **Main-line and spouse-side depth are not comparable without design context.** Family records explicitly distinguish `main_line`, `spouse_ancestry`, and `collateral`; minimal spouse branches reflect scope choices as well as research coverage.
4. **Household and fertility measures must retain “recorded.”** A zero-child family record or terminal person is not evidence of historical childlessness or biological extinction.
5. **The next data-model improvement is explicit status metadata.** Birthplace, residence, sex, marriage date, privacy/living state, and value-state categories (unknown / not applicable / inferred / recorded) would make several requested analyses possible without proxies.

### Reproducibility notes

- Rebuild the source database with `uv run python -m scripts.build_genealogy_analytics` after registry changes, then rerun this notebook top to bottom.
- Plotly legends are clickable to show/hide series. Hover labels identify marks and report axis values; they disappear when the pointer leaves. Time-series axes are continuous and use alternating decade washes.
- Chart subtitles expose date-based denominators. Bar axes start at zero; percentages use consistent one-decimal precision.
- This notebook closes its read-only connection in the final cell.

In [28]:
con.close()
print('Read-only DuckDB connection closed.')

Read-only DuckDB connection closed.
